# Suasa 2023 Survey — Statistical Analysis (clean version, 13 September 2026)

Notebook reorganised from earlier versions (7 February, 10 February and 21 July 2025),
correcting the issues identified during review:

1. **Surveyor encoding**: in earlier versions the regression sometimes used `Ricognitor_code`
   (an arbitrary ordinal integer obtained with `.cat.codes`), which implies an ordering among
   surveyors that has no meaning. Here the surveyor is always treated as a categorical variable
   (`C(...)` in the model formula, equivalent to one-hot encoding with a baseline level).
2. **Visibility in the multivariate model**: entering `vis` as a continuous variable with a
   quadratic term (`vis + vis^2`) does not work with only three distinct levels (3, 4, 5): it
   produces strong collinearity (Condition Number > 700) and coefficients without statistical
   power. `vis` is therefore also treated as categorical (`C(vis_category)`), with 'Fair' as the
   baseline level.
3. **Whole dataset vs. restricted area**: the two analyses are kept separate and explicitly
   labelled, to avoid mixing patterns that go in different directions in the two subsets.
4. **Weather as a possible confounder**: raw visibility, uncontrolled for weather, suggested a
   peak at intermediate visibility ('Good') in the restricted area. When weather is controlled
   for in the multivariate model, the pattern simplifies to a monotonic effect in the point
   estimates (higher visibility, more finds), with only the 'Excellent' level significantly
   different from the baseline level 'Fair'.

**Before running**: replace the two paths in the configuration cell with your own.
If `fiona` raises an error (`AttributeError: module 'fiona' has no attribute 'path'`), check that
the intended Python environment is selected, or pass `engine="pyogrio"` to `gpd.read_file(...)`.

**Data availability**: **[TODO: add the repository / DOI of the shapefiles used here.]**

### Fields used in this notebook

The field names below are those stored in the shapefiles and are kept unchanged, so that the
code runs on the original data.

| Field | Meaning |
|---|---|
| `Ricognitor` | Surveyor (field walker) who recorded the point. |
| `vis` | Ground visibility, ordinal score 1–5 (1 None, 2 Poor, 3 Fair, 4 Good, 5 Excellent). Only levels 3–5 occur in this dataset. |
| `MAT` | Bricks (*Mattoni*). |
| `CCom` | Common ware (*ceramica comune*). |
| `dens_MAT`, `dens_CCom` | Density of `MAT` and `CCom`. |
| `Meteo` | Weather during the survey: `Coperto` (overcast), `Sereno` (clear), `Non specif` (not specified). |

## 1. Configuration and imports

In [ ]:
# Input paths -- EDIT HERE. Relative paths are resolved from the notebook's working directory.
PATH_POINTS = "SHP/UT_lastunion/UT_union_noduplicate_centroide.shp"           # survey points
PATH_RESTRICTED_AREA = "SHP/UT_lastunion/area_restrizione_indagine_pol.shp"   # polygon of the hypothesised villa

# Visibility scale recorded in the field (1-5). Labels are translated from the original Italian
# (Nulla, Scarsa, Discreta, Buona, Ottima).
VISIBILITY_MAP = {1: "None", 2: "Poor", 3: "Fair", 4: "Good", 5: "Excellent"}

# Levels present in the data (3-5), in ascending order. This list fixes the order of tables and
# plots and, in the regression of Section 7, makes 'Fair' (the lowest level) the baseline.
VISIBILITY_ORDER = ["Fair", "Good", "Excellent"]

In [ ]:
import geopandas as gpd                        # reading shapefiles and spatial operations
import pandas as pd                            # tabular data handling
import numpy as np
import matplotlib.pyplot as plt                # plotting
import seaborn as sns                          # boxplots
from scipy.stats import shapiro, levene, kruskal   # normality test and non-parametric group comparison
import statsmodels.formula.api as smf          # OLS regression with R-style formulas
import scikit_posthocs as sp                   # Dunn post-hoc test after Kruskal-Wallis

sns.set_style("whitegrid")
%matplotlib inline

## 2. Data loading and cleaning

In [ ]:
# Read the survey points. If gpd.read_file raises an AttributeError related to fiona,
# comment out the first line and uncomment the one with engine="pyogrio".
gdf = gpd.read_file(PATH_POINTS)
# gdf = gpd.read_file(PATH_POINTS, engine="pyogrio")

# Numeric attributes sometimes arrive from the shapefile as strings. Force them to numbers so that
# means, tests and models work; errors="coerce" turns unparseable values into NaN, and the rows
# concerned are dropped by dropna() in each analysis below.
for col in ["MAT", "CCom", "vis", "dens_MAT", "dens_CCom"]:
    if col in gdf.columns:
        gdf[col] = pd.to_numeric(gdf[col], errors="coerce")

# Text label for each visibility score: the labels are what the categorical treatment
# (tables, plots, model formula) relies on.
gdf["vis_category"] = gdf["vis"].map(VISIBILITY_MAP)

print("Total rows:", len(gdf))
gdf[["Ricognitor", "vis", "vis_category", "MAT", "CCom", "Meteo"]].head()

In [ ]:
# Polygon delimiting the area of the hypothesised villa
restricted_area = gpd.read_file(PATH_RESTRICTED_AREA)
# restricted_area = gpd.read_file(PATH_RESTRICTED_AREA, engine="pyogrio")

# Points and polygon must share one CRS, otherwise the spatial selection below is meaningless.
gdf = gdf.to_crs(restricted_area.crs)

# Merge all polygon features into one geometry, so that "within" is evaluated against the whole
# area at once. union_all() replaces the deprecated unary_union (geopandas >= 1.0); the fallback
# keeps older versions working.
if hasattr(restricted_area, "union_all"):
    restricted_area_geom = restricted_area.union_all()
else:
    restricted_area_geom = restricted_area.unary_union

# Spatial subset used in Sections 5, 7 and 9. Points lying exactly on the boundary are not
# "within" the polygon and are therefore excluded.
gdf_restricted = gdf[gdf.geometry.within(restricted_area_geom)].copy()

print(f"Total points: {len(gdf)}")
print(f"Points inside the restricted area (hypothesised villa): {len(gdf_restricted)}")

In [ ]:
# Visual check that the spatial selection behaves as expected
fig, ax = plt.subplots(figsize=(9, 7))
gdf_restricted.plot(marker="o", color="steelblue", markersize=15, ax=ax, label="Points")
restricted_area.plot(edgecolor="red", facecolor="none", linestyle="--", linewidth=2, ax=ax, label="Restricted area")
ax.set_title("Points inside the restricted area (suburban villa hypothesis)")
# The axis labels assume the layers are in EPSG:32633; update them if your CRS differs.
ax.set_xlabel("X coordinate (EPSG:32633)")
ax.set_ylabel("Y coordinate (EPSG:32633)")
plt.tight_layout()
plt.show()

## 3. Surveyor bias — MAT by surveyor

Non-parametric test (Kruskal-Wallis) on the whole dataset, because the MAT distributions are not
normal (see the Shapiro-Wilk test below). All tests use a significance threshold of 0.05.

In [ ]:
df_mat = gdf.dropna(subset=["Ricognitor", "MAT"])
# "Other" is excluded: it pools different surveyors who cannot be told apart, so it does not
# represent a single operator.
df_mat = df_mat[df_mat["Ricognitor"] != "Other"]

# Shapiro-Wilk test within each surveyor group: it motivates the use of a non-parametric test.
print("Shapiro-Wilk test for MAT, by surveyor:")
for name, group_values in df_mat.groupby("Ricognitor")["MAT"]:
    if len(group_values) >= 3:      # the test needs at least 3 observations
        stat, p = shapiro(group_values)
        outcome = "normality not rejected" if p > 0.05 else "normality rejected"
        print(f"  {name}: stat={stat:.4f}, p={p:.4f} -> {outcome}")
    else:
        print(f"  {name}: only {len(group_values)} observations, test not applicable")

In [ ]:
# Kruskal-Wallis: compares the distributions of MAT across the surveyors without assuming normality.
groups_mat = [g["MAT"].values for _, g in df_mat.groupby("Ricognitor")]
h_mat, p_mat = kruskal(*groups_mat)
dof_mat = len(groups_mat) - 1       # degrees of freedom = number of groups - 1

print(f"Kruskal-Wallis MAT by surveyor: H({dof_mat}) = {h_mat:.4f}, p = {p_mat:.4f}")
print("=> significant difference between surveyors" if p_mat < 0.05
      else "=> NO significant difference between surveyors")

# Group means, sorted in descending order (also used to order the boxplot below).
mean_mat = df_mat.groupby("Ricognitor")["MAT"].mean().sort_values(ascending=False)
print("\nMean MAT by surveyor:")
print(mean_mat)

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x="Ricognitor", y="MAT", data=df_mat, order=mean_mat.index)
plt.title("Distribution of MAT by surveyor")
plt.tight_layout()
plt.show()

## 4. Visibility and common ware (CCom) — whole dataset

**Note**: this is the analysis of the whole survey, distinct from the one on the restricted area
(Section 5). The two subsets show different patterns: they should neither be confused nor
pooled.

In [ ]:
df_ccom = gdf.dropna(subset=["vis_category", "CCom"])

# Descriptive statistics per visibility level (reindex sets the order Fair -> Good -> Excellent)
stats_ccom = df_ccom.groupby("vis_category")["CCom"].agg(["count", "mean", "median", "std"])
stats_ccom = stats_ccom.reindex(VISIBILITY_ORDER)
print(stats_ccom)

In [ ]:
# Kruskal-Wallis across the three visibility levels (non-parametric, as in Section 3)
groups_vis = [df_ccom[df_ccom["vis_category"] == v]["CCom"].values for v in VISIBILITY_ORDER]
h_vis, p_vis = kruskal(*groups_vis)
dof_vis = len(groups_vis) - 1

print(f"Kruskal-Wallis CCom by visibility (whole dataset): H({dof_vis}) = {h_vis:.4f}, p = {p_vis:.6f}")

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(x="vis_category", y="CCom", data=df_ccom, order=VISIBILITY_ORDER)
plt.title("CCom by visibility level - whole dataset")
plt.xlabel("Visibility")
plt.tight_layout()
plt.show()

## 5. Visibility and common ware (CCom) — restricted area only (hypothesised villa)

Same analysis as Section 4, restricted to the area of the hypothesised villa, followed by the raw
Dunn post-hoc test (not controlled for weather or surveyor). The Dunn result is the reference
for comparison with the multivariate model of Section 7.

In [ ]:
df_ccom_r = gdf_restricted.dropna(subset=["vis_category", "CCom"])
stats_ccom_r = df_ccom_r.groupby("vis_category")["CCom"].agg(["count", "mean", "median", "std"])
stats_ccom_r = stats_ccom_r.reindex(VISIBILITY_ORDER)
print(stats_ccom_r)

In [ ]:
groups_vis_r = [df_ccom_r[df_ccom_r["vis_category"] == v]["CCom"].values for v in VISIBILITY_ORDER]
h_vis_r, p_vis_r = kruskal(*groups_vis_r)
dof_vis_r = len(groups_vis_r) - 1

print(f"Kruskal-Wallis CCom by visibility (restricted area): H({dof_vis_r}) = {h_vis_r:.4f}, p = {p_vis_r:.6f}")

# The post-hoc test is run only if Kruskal-Wallis is significant. Dunn's test compares the groups
# pairwise; the Bonferroni correction accounts for the multiple comparisons.
if p_vis_r < 0.05:
    dunn_r = sp.posthoc_dunn(df_ccom_r, val_col="CCom", group_col="vis_category", p_adjust="bonferroni")
    dunn_r = dunn_r.reindex(index=VISIBILITY_ORDER, columns=VISIBILITY_ORDER)
    print("\nDunn post-hoc test (raw, not controlled for weather/surveyor):")
    print(dunn_r)

In [ ]:
# Side-by-side comparison of the two subsets; sharey=True keeps the y-axis identical so that the
# two panels are directly comparable.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
sns.boxplot(x="vis_category", y="CCom", data=df_ccom, order=VISIBILITY_ORDER, ax=axes[0])
axes[0].set_title("Whole dataset")
sns.boxplot(x="vis_category", y="CCom", data=df_ccom_r, order=VISIBILITY_ORDER, ax=axes[1])
axes[1].set_title("Restricted area (hypothesised villa)")
for ax in axes:
    ax.set_xlabel("Visibility")
plt.suptitle("CCom by visibility level - comparison between the two subsets")
plt.tight_layout()
plt.savefig("ccom_visibility_comparison.png", dpi=150)
plt.show()

## 6. Weather conditions and common ware (CCom)

Whole dataset. `Non specif` (weather not specified) is kept as a category of its own rather than
dropped, so it appears alongside `Coperto` and `Sereno` in the tables and in the model of Section 7.

In [ ]:
df_weather = gdf.dropna(subset=["Meteo", "CCom"])
stats_weather = df_weather.groupby("Meteo")["CCom"].agg(["count", "mean", "median", "std"])
print(stats_weather)

In [ ]:
groups_weather = [g["CCom"].values for _, g in df_weather.groupby("Meteo")]
h_weather, p_weather = kruskal(*groups_weather)
dof_weather = len(groups_weather) - 1

print(f"Kruskal-Wallis CCom by weather: H({dof_weather}) = {h_weather:.4f}, p = {p_weather:.6f}")

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(x="Meteo", y="CCom", data=df_weather)
plt.title("CCom by weather conditions")
plt.tight_layout()
plt.show()

## 7. Multivariate model — CCom density by visibility, weather and surveyor

**Correction with respect to earlier versions**: `vis` and `Ricognitor` are treated as categorical
variables (`C(...)`), not as ordinal numbers or as a continuous quadratic term (the latter,
tested in an earlier version, was unstable because of collinearity: Condition Number > 700, with
only three distinct visibility levels).

Dataset: restricted area (hypothesised villa).

In [ ]:
# Keep only rows with all four variables of the model (OLS would drop them silently otherwise).
ccom_data = gdf_restricted.dropna(subset=["dens_CCom", "vis_category", "Ricognitor", "Meteo"]).copy()

# Fix the category order explicitly, so that the model baseline is 'Fair' (the lowest level)
# rather than an arbitrary alphabetical choice. ordered=False: the order only selects the
# baseline; the model does not treat visibility as an ordinal scale.
ccom_data["vis_category"] = pd.Categorical(ccom_data["vis_category"], categories=VISIBILITY_ORDER, ordered=False)

# OLS with three categorical predictors. C(...) expands each variable into dummy variables
# (treatment coding): one coefficient per non-baseline level.
model_ccom = smf.ols("dens_CCom ~ C(vis_category) + C(Meteo) + C(Ricognitor)", data=ccom_data).fit()
print(model_ccom.summary())

**How to read the model**: the baseline is Visibility = Fair, Weather = Coperto, Surveyor = Andrea
(the first level in alphabetical order for Meteo and Ricognitor, since no explicit order was
fixed for these two, unlike visibility).

- The coefficient of each dummy `C(vis_category)[T.X]` is the expected difference in `dens_CCom`
  between level X and the baseline (Fair), holding weather and surveyor constant.
- If no `C(Ricognitor)` dummy is significant, no individual surveyor differs detectably from the
  baseline surveyor. A joint test over all surveyor dummies (e.g. `model_ccom.wald_test`) is not
  run here. With an ordinal encoding the picture could have looked different (bug corrected in
  this version).
- Compare the coefficient and p-value of `C(vis_category)[T.Excellent]` with the raw Dunn test of
  Section 5. If the pattern changes (e.g. the peak shifts, or a difference that was significant
  is no longer so), this suggests that weather or surveyor acted as confounders in the
  univariate analysis.

## 8. Summary

Run this cell at the end to collect all the key numbers in one place, for reporting in the text
of the article or talk. It relies on the variables created in Sections 3–7.

In [ ]:
print("SUMMARY OF RESULTS")
print("=" * 60)
print(f"1) MAT by surveyor (KW, whole dataset):")
print(f"   H({dof_mat}) = {h_mat:.2f}, p = {p_mat:.4f}")
print()
print(f"2) CCom by visibility (KW, whole dataset):")
print(f"   H({dof_vis}) = {h_vis:.2f}, p = {p_vis:.6f}")
print(f"   Means: {stats_ccom['mean'].round(1).to_dict()}")
print()
print(f"3) CCom by visibility (KW, restricted area):")
print(f"   H({dof_vis_r}) = {h_vis_r:.2f}, p = {p_vis_r:.6f}")
print(f"   Means: {stats_ccom_r['mean'].round(1).to_dict()}")
print()
print(f"4) CCom by weather (KW, whole dataset):")
print(f"   H({dof_weather}) = {h_weather:.2f}, p = {p_weather:.6f}")
print()
print(f"5) Multivariate model (dens_CCom ~ visibility + weather + surveyor, restricted area):")
print(f"   R^2 = {model_ccom.rsquared:.3f}, F p-value = {model_ccom.f_pvalue:.2e}")
for term, coef in model_ccom.params.items():
    if term != "Intercept":
        print(f"   {term}: coef={coef:.4f}, p={model_ccom.pvalues[term]:.4f}")

## 9. Supplementary analyses

These two analyses were appended after the summary and are not included in it.

### 9.1 MAT density — multivariate model, whole dataset

Same model structure as Section 7, applied to `dens_MAT` on the whole dataset ("Other"
surveyors excluded, as in Section 3).

In [ ]:
mat_data = gdf.dropna(subset=["dens_MAT", "vis_category", "Ricognitor", "Meteo"]).copy()
mat_data = mat_data[mat_data["Ricognitor"] != "Other"]    # same exclusion as in Section 3

# Baseline for visibility. The original run did not fix a category order here, so statsmodels
# used the alphabetically first level, 'Buona' in the Italian labels, i.e. 'Good'. 'Good' is set
# explicitly to reproduce those results after the translation of the labels.
# NOTE: this differs from Section 7, where the baseline is 'Fair'. To make the two models
# comparable, replace the list below with VISIBILITY_ORDER.
mat_data["vis_category"] = pd.Categorical(mat_data["vis_category"],
                                          categories=["Good", "Fair", "Excellent"], ordered=False)

model_mat = smf.ols("dens_MAT ~ C(vis_category) + C(Meteo) + C(Ricognitor)", data=mat_data).fit()
print(model_mat.summary())

### 9.2 Weather and CCom — restricted area only

Descriptive statistics and Kruskal-Wallis test as in Section 6, on the restricted area.

In [ ]:
df_weather_r = gdf_restricted.dropna(subset=["Meteo", "CCom"])
stats_weather_r = df_weather_r.groupby("Meteo")["CCom"].agg(["count", "mean", "median", "std"])
print(stats_weather_r)

groups_weather_r = [g["CCom"].values for _, g in df_weather_r.groupby("Meteo")]
h_weather_r, p_weather_r = kruskal(*groups_weather_r)
print(f"\nKruskal-Wallis CCom by weather (restricted area): H({len(groups_weather_r) - 1}) = {h_weather_r:.4f}, p = {p_weather_r:.6f}")